<a href="https://colab.research.google.com/github/aapaul29/Distance-Recognition-SML1-/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SML Project 1
This notebook works in both **VS Code (local)** and **Google Colab**.

## 1. Environment Detection & Setup

In [454]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
print(f"Running in {'Google Colab' if IN_COLAB else 'VS Code / local'}")

Running in VS Code / local


## 2. Google Colab: Mount Drive & Navigate to Project
This cell only runs on Colab. Adjust `COLAB_PROJECT_PATH` to where you uploaded the project folder in your Google Drive.

In [455]:
# ---------------------------------------------------------------
# COLAB ONLY — skip if running locally
# Adjust this path to match your Google Drive folder structure.
COLAB_PROJECT_PATH = "/content/drive/MyDrive/Aaryan/Uni/BS Mech Eng ETH Zurich/2nd year/Semester 4/Stochastics & Machine Learning/Distance-Recognition-SML1-"
# ---------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd $COLAB_PROJECT_PATH
    print(f"Working directory: {os.getcwd()}")

    # Unzip data if not already extracted
    data_zip = os.path.join(COLAB_PROJECT_PATH, "data.zip")
    data_dir = os.path.join(COLAB_PROJECT_PATH, "data")
    if os.path.exists(data_zip) and not os.path.exists(data_dir):
        print("Unzipping data...")
        os.makedirs(data_dir, exist_ok=True)
        !unzip -q $data_zip -d $data_dir
        print("Done.")
    else:
        print("Data directory already exists or no zip found — skipping unzip.")
else:
    print("Local environment — no mounting needed.")

Local environment — no mounting needed.


## 3. Install / Verify Dependencies

In [456]:
# On Colab, install any missing packages from requirements.txt.
# Locally, this is a no-op if your conda env is already set up.
if IN_COLAB:
    !pip install -q -r requirements.txt

## 4. Auto-reload & Imports

In [457]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from utils import load_config, load_dataset, load_test_dataset, print_results, save_results

# Add further sklearn imports here as needed.
# Note: SVRs are NOT allowed in this project.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 5. Load Config & Data

In [458]:
config = load_config()

images, distances = load_dataset(config, split="train")
print(f"[INFO]: Dataset loaded with {len(images)} samples.")
print(f"[INFO]: Feature dimensionality: {images.shape[1]}")

[INFO]: Configs are loaded with: 
 {'data_dir': PosixPath('data'), 'load_rgb': False, 'downsample_factor': 3}
[INFO]: Dataset loaded with 3000 samples.
[INFO]: Feature dimensionality: 10000


## 6. Your Implementation
Implement preprocessing, model training, and evaluation below.

In [459]:
# TODO: Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import cv2

ds = config["downsample_factor"]
H = W = 300 // ds
C = 3 if config["load_rgb"] else 1

def get_pca_features(X, H, W):
    """Model 1: Rohe Pixel + gewichtet → PCA"""
    weights = np.linspace(0.5, 1.5, H).reshape(H, 1)
    all_feat = []
    for img in X:
        img_2d = img.reshape(H, W).astype(np.float32)
        img_weighted = (img_2d * weights).astype(np.uint8)
        all_feat.append(img_weighted.reshape(-1).astype(np.float32))
    return np.array(all_feat)

def get_grid_stats_features(X, H, W):
    """Model 2: Grid + Stats + Histogram"""
    all_feat = []
    for img in X:
        img_2d = img.reshape(H, W).astype(np.uint8)

        # Grid 4x4
        grid_features = []
        grid_h, grid_w = H // 4, W // 4
        for i in range(4):
            for j in range(4):
                cell = img_2d[i*grid_h:(i+1)*grid_h, j*grid_w:(j+1)*grid_w]
                grid_features.extend([cell.mean(), cell.var()])

        # Horizontale Streifen
        strip_features = []
        strip_h = H // 8
        for i in range(8):
            strip = img_2d[i*strip_h:(i+1)*strip_h, :]
            strip_features.extend([strip.mean(), strip.var()])

        # Global + Histogram
        hist, _ = np.histogram(img_2d, bins=16, range=(0, 255))
        hist = hist.astype(np.float32) / hist.sum()
        global_feat = np.array([img_2d.mean(), img_2d.std()])
        col_min = np.min(img_2d, axis=0).astype(np.float32)

        all_feat.append(np.concatenate([grid_features, strip_features, hist, global_feat, col_min]))
    return np.array(all_feat)

def get_edge_features(X, H, W):
    """Model 3: Edge + Gradient + Depth Proxy"""
    all_feat = []
    for img in X:
        img_2d = img.reshape(H, W).astype(np.uint8)

        # Sobel separat
        sobel_x = cv2.Sobel(img_2d, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(img_2d, cv2.CV_64F, 0, 1, ksize=3)
        hist_sx, _ = np.histogram(sobel_x, bins=16)
        hist_sy, _ = np.histogram(sobel_y, bins=16)
        hist_sx = hist_sx.astype(np.float32) / (hist_sx.sum() + 1e-8)
        hist_sy = hist_sy.astype(np.float32) / (hist_sy.sum() + 1e-8)
        sobel_feat = np.array([sobel_x.mean(), sobel_x.var(), sobel_y.mean(), sobel_y.var()])

        # Floor Scan
        floor_scan = np.zeros(W, dtype=np.float32)
        for col in range(W):
            column = img_2d[:, col].astype(np.float32)
            grad = np.abs(np.diff(column))
            for row in range(H-2, 0, -1):
                if grad[row-1] > 15:
                    floor_scan[col] = row / H
                    break

        # Streifen Gradients
        strip_grad = []
        strip_h = H // 8
        for i in range(8):
            strip = img_2d[i*strip_h:(i+1)*strip_h, :]
            sx = cv2.Sobel(strip, cv2.CV_64F, 1, 0, ksize=3)
            sy = cv2.Sobel(strip, cv2.CV_64F, 0, 1, ksize=3)
            mag = np.sqrt(sx**2 + sy**2)
            strip_grad.extend([mag.mean(), mag.var()])

        all_feat.append(np.concatenate([sobel_feat, hist_sx, hist_sy, floor_scan, strip_grad]))
    return np.array(all_feat)

def augment(X, y, H, W, C):
    X_aug, y_aug = [X], [y]
    X_flip = X.reshape(-1, H, W, C)[:, :, ::-1, :].reshape(len(X), -1)
    X_aug.append(X_flip); y_aug.append(y)
    X_dark = np.clip(X * 0.8, 0, 255)
    X_aug.append(X_dark); y_aug.append(y)
    X_bright = np.clip(X * 1.2, 0, 255)
    X_aug.append(X_bright); y_aug.append(y)
    X_blur = []
    kernel = np.ones((3, 3), np.float32) / 9
    for img in X:
        img_2d = img.reshape(H, W).astype(np.uint8)
        blurred = cv2.filter2D(img_2d, -1, kernel).reshape(-1)
        X_blur.append(blurred)
    X_aug.append(np.array(X_blur)); y_aug.append(y)
    X_erase = []
    for img in X:
        img_2d = img.reshape(H, W).copy()
        x1, y1 = np.random.randint(0, W-10), np.random.randint(0, H-10)
        img_2d[y1:y1+10, x1:x1+10] = 0
        X_erase.append(img_2d.reshape(-1))
    X_aug.append(np.array(X_erase)); y_aug.append(y)
    return np.vstack(X_aug), np.concatenate(y_aug)

# Augmentierung
X_aug, y_aug = augment(images, distances, H, W, C)

# Split zuerst
X_train_raw, X_val_raw, y_train, y_val = train_test_split(X_aug, y_aug, test_size=0.15, random_state=42)

# Feature Set 1: PCA
X1_train = get_pca_features(X_train_raw, H, W)
X1_val = get_pca_features(X_val_raw, H, W)
scaler1 = StandardScaler()
X1_train = scaler1.fit_transform(X1_train)
X1_val = scaler1.transform(X1_val)
pca = PCA(n_components=100)
X1_train = pca.fit_transform(X1_train)
X1_val = pca.transform(X1_val)

# Feature Set 2: Grid+Stats
X2_train = get_grid_stats_features(X_train_raw, H, W)
X2_val = get_grid_stats_features(X_val_raw, H, W)
scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train)
X2_val = scaler2.transform(X2_val)

# Feature Set 3: Edge+Depth
X3_train = get_edge_features(X_train_raw, H, W)
X3_val = get_edge_features(X_val_raw, H, W)
scaler3 = StandardScaler()
X3_train = scaler3.fit_transform(X3_train)
X3_val = scaler3.transform(X3_val)

print(f"Set 1: {X1_train.shape}, Set 2: {X2_train.shape}, Set 3: {X3_train.shape}")

Set 1: (15300, 100), Set 2: (15300, 166), Set 3: (15300, 152)


In [460]:
# TODO: Model definition & training
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
import numpy as np

# Jedes Modell bekommt sein eigenes Feature-Set
# Wir kombinieren alle 3 Feature-Sets
X_train_all = np.hstack([X1_train, X2_train, X3_train])
X_val_all = np.hstack([X1_val, X2_val, X3_val])

estimators = [
    ("hgb", HistGradientBoostingRegressor(max_iter=500, max_depth=7, learning_rate=0.05, random_state=42)),
    ("et", ExtraTreesRegressor(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)),
    ("gb", GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42)),
]

model = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1)

print("Training läuft...")
model.fit(X_train_all, y_train)
print("Fertig!")

Training läuft...
Fertig!


In [461]:
# TODO: Evaluation on training / validation set
print("Validation:")
print_results(y_val, model.predict(X_val_all))
print("Training:")
print_results(y_train, model.predict(X_train_all))

Validation:
MAE: 9.502
R2: 89.011
Training:
MAE: 4.716
R2: 97.842


## 7. Predict on Test Set & Save

In [462]:
test_images_raw = load_test_dataset(config)
print(f"[INFO]: Test set loaded with {len(test_images_raw)} samples.")

X_test_raw = np.array(test_images_raw)
X1_test = pca.transform(scaler1.transform(get_pca_features(X_test_raw, H, W)))
X2_test = scaler2.transform(get_grid_stats_features(X_test_raw, H, W))
X3_test = scaler3.transform(get_edge_features(X_test_raw, H, W))
X_test_all = np.hstack([X1_test, X2_test, X3_test])

test_pred = model.predict(X_test_all)
save_results(test_pred)
print("Gespeichert!")

[INFO]: Test set loaded with 622 samples.
Gespeichert!
